# Virtual Try-On - CatVTON GPU Backend
## Cell 1: Install Dependencies

In [ ]:
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON
!pip install -q diffusers accelerate transformers peft huggingface_hub av
!pip install -q fastapi uvicorn python-multipart nest_asyncio
!pip install -q opencv-python-headless
!pip install -q fvcore iopath
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q -U cloudflared
print("✓ All dependencies installed!")

## Cell 2: Load Model

In [ ]:
import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, ".")

import torch
import numpy as np
import cv2
from PIL import Image, ImageEnhance
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download

from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

WIDTH = 768
HEIGHT = 1024
DEVICE = "cuda"

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
print(f"✓ Model downloaded to: {repo_path}")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device=DEVICE,
    skip_safety_check=True,
)

mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True
)

automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=DEVICE
)

print(f"✓ Models loaded! Resolution: {WIDTH}x{HEIGHT}")
print(f"✓ VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Cell 3: Helper Functions

In [ ]:
def sharpen_image(image, amount=1.3):
    """Apply sharpening to reduce blur"""
    enhancer = ImageEnhance.Sharpness(image)
    return enhancer.enhance(amount)

print("✓ Helper functions loaded")

## Cell 4: API Server

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio
from io import BytesIO
import base64
import time
import gc

nest_asyncio.apply()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
async def health_check():
    return {
        "status": "ok",
        "gpu": torch.cuda.get_device_name(0),
        "resolution": f"{WIDTH}x{HEIGHT}"
    }

@app.post("/api/try-on")
async def try_on_endpoint(
    person_image: UploadFile = File(...),
    cloth_image: UploadFile = File(...),
    cloth_type: str = Form("upper"),
    num_inference_steps: int = Form(50),
    guidance_scale: float = Form(4.0),
    seed: int = Form(42),
):
    try:
        start_time = time.time()

        # Load images
        person = Image.open(BytesIO(await person_image.read())).convert("RGB")
        garment = Image.open(BytesIO(await cloth_image.read())).convert("RGB")

        print(f"→ Loaded: Person {person.size}, Garment {garment.size}")

        # Resize
        person = resize_and_crop(person, (WIDTH, HEIGHT))
        garment = resize_and_padding(garment, (WIDTH, HEIGHT))

        # Generate mask
        print(f"→ Generating mask: {cloth_type}")
        mask_result = automasker(person, cloth_type)
        
        # Handle different return types from automasker
        if isinstance(mask_result, dict):
            mask = mask_result.get('mask', list(mask_result.values())[0])
        elif isinstance(mask_result, (list, tuple)):
            mask = mask_result[0]
        else:
            mask = mask_result
            
        mask = mask_processor.blur(mask, blur_factor=9)

        # Run inference
        print(f"→ Running inference: {num_inference_steps} steps, guidance {guidance_scale}")

        with torch.inference_mode():
            result = pipeline(
                image=person,
                condition_image=garment,
                mask=mask,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=torch.Generator(DEVICE).manual_seed(seed) if seed >= 0 else None,
            )[0]

        # Sharpen
        result = sharpen_image(result)

        # Convert to base64
        buffered = BytesIO()
        result.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()

        # Cleanup
        gc.collect()
        torch.cuda.empty_cache()

        elapsed = round(time.time() - start_time, 1)
        print(f"✓ Completed in {elapsed}s\n")

        return JSONResponse({
            "status": "success",
            "result_image": img_base64,
            "elapsed_seconds": elapsed
        })

    except Exception as e:
        print(f"✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return JSONResponse(
            status_code=500,
            content={"status": "error", "detail": str(e)}
        )

print("✓ API server configured")

## Cell 5: Start Server & Tunnel

In [ ]:
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("\n" + "="*60)
print("  Starting server...")
print("="*60)

import time
time.sleep(3)

print("✓ Server started on port 8000\n")
print("="*60)
print("  Starting Cloudflare tunnel...")
print("="*60)

# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Start Cloudflare tunnel
!cloudflared tunnel --url http://localhost:8000

# The tunnel URL will appear above (https://xxxxx.trycloudflare.com)
# Copy that URL and paste it into your web app!